# NumPy Dojo — Block 2: Activations & Losses

Dataset: **synthetic logits and labels** — dimensioned to mimic a real 10-class classifier.

Rules:
- No torch, no scipy — only numpy
- Every function must handle **batched inputs** (2D arrays where axis=0 is batch)
- Implement gradients too — this is what separates prep from production

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

N, C = 64, 10  # batch of 64, 10 classes
logits = np.random.randn(N, C) * 3.0          # simulates raw classifier output
labels = np.random.randint(0, C, size=(N,))    # ground truth class indices

# For binary tasks
N_bin = 128
logits_bin = np.random.randn(N_bin)            # (N,) raw binary logits
labels_bin  = np.random.randint(0, 2, size=(N_bin,)).astype(np.float64)

# For regression
y_pred = np.random.randn(N)
y_true = np.random.randn(N)

print(f'logits: {logits.shape}, labels: {labels.shape}')
print(f'logits range: [{logits.min():.2f}, {logits.max():.2f}]')

---
## P5 — Activation Functions

Implement each activation and its gradient. Input `x` is any shape — output must be same shape.

Functions to implement:
- ReLU and its gradient
- Leaky ReLU (alpha=0.01) and its gradient
- Sigmoid and its gradient
- Tanh (using `np.exp` directly, not `np.tanh`) and its gradient
- GELU (approximate): `0.5 * x * (1 + tanh(√(2/π) * (x + 0.044715 * x³)))`
- Swish: `x * sigmoid(x)` — use your sigmoid

In [ ]:
def relu(x):
    # TODO
    pass

def relu_grad(x):
    """Derivative of ReLU w.r.t. x"""
    # TODO: returns 1 where x>0, else 0
    pass

def leaky_relu(x, alpha=0.01):
    # TODO
    pass

def leaky_relu_grad(x, alpha=0.01):
    # TODO
    pass

def sigmoid(x):
    # TODO — watch out for overflow on large negative x
    pass

def sigmoid_grad(x):
    """ds/dx = s(x) * (1 - s(x))"""
    # TODO
    pass

def tanh_manual(x):
    """Implement tanh from scratch using exp — not np.tanh"""
    # TODO: tanh(x) = (e^x - e^-x) / (e^x + e^-x)
    # Think about numerical stability for large x
    pass

def tanh_grad(x):
    """Derivative: 1 - tanh²(x)"""
    # TODO
    pass

def gelu(x):
    # TODO
    pass

def swish(x):
    # TODO: use your sigmoid
    pass

In [ ]:
# --- ASSERTS ---
x_test = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

# ReLU
assert np.allclose(relu(x_test), [0, 0, 0, 1, 2]), 'relu failed'
assert np.allclose(relu_grad(x_test), [0, 0, 0, 1, 1]), 'relu_grad failed'

# Leaky ReLU
assert np.allclose(leaky_relu(x_test), [-0.02, -0.01, 0, 1, 2]), 'leaky_relu failed'

# Sigmoid
assert np.allclose(sigmoid(np.array([0.0])), [0.5]), 'sigmoid(0) != 0.5'
assert np.allclose(sigmoid(np.array([500.0])), [1.0], atol=1e-6), 'sigmoid overflow'
assert np.allclose(sigmoid(np.array([-500.0])), [0.0], atol=1e-6), 'sigmoid underflow'

# Sigmoid gradient consistency with sigmoid values
s = sigmoid(x_test)
sg = sigmoid_grad(x_test)
assert np.allclose(sg, s * (1 - s)), 'sigmoid_grad formula wrong'

# Tanh vs numpy reference
assert np.allclose(tanh_manual(x_test), np.tanh(x_test), atol=1e-6), 'tanh_manual wrong'

# Tanh gradient consistency
tg = tanh_grad(x_test)
assert np.allclose(tg, 1 - tanh_manual(x_test)**2), 'tanh_grad formula wrong'

# Shape preservation
for fn in [relu, leaky_relu, sigmoid, tanh_manual, gelu, swish]:
    assert fn(logits).shape == logits.shape, f'{fn.__name__} changed shape'

print('P5 PASSED ✓')

In [ ]:
# Visualize all activations
x_vis = np.linspace(-4, 4, 300)
fns = [
    ('ReLU', relu), ('Leaky ReLU', leaky_relu),
    ('Sigmoid', sigmoid), ('Tanh', tanh_manual),
    ('GELU', gelu), ('Swish', swish)
]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, fn) in zip(axes.flat, fns):
    ax.plot(x_vis, fn(x_vis), linewidth=2)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(name)
    ax.set_ylim(-2, 3)
    ax.grid(alpha=0.3)
plt.suptitle('Activation Functions')
plt.tight_layout()
plt.show()

---
## P6 — Softmax (Stable) + Categorical Cross-Entropy

Input: `logits` of shape **(N, C)**, `labels` of shape **(N,)** — integer class indices.

Implement:
1. Numerically stable softmax (row-wise)
2. Mean cross-entropy loss
3. Top-1 accuracy
4. Gradient of loss w.r.t. logits (the clean formula: `softmax(z) - one_hot(y)`)

Then implement the **unstable** softmax and show it breaks on large logits.

In [ ]:
def softmax(z):
    """
    z: (N, C) or (C,)
    Returns probabilities, same shape
    """
    # TODO: subtract row-wise max before exponentiating
    pass

def softmax_unstable(z):
    """Naive softmax — no max subtraction"""
    # TODO
    pass

def cross_entropy_loss(logits, labels):
    """
    logits: (N, C)
    labels: (N,) integer class indices
    Returns: scalar mean loss
    """
    # TODO: softmax -> pick correct class prob -> log -> mean
    # Hint: advanced indexing — pick one column per row
    pass

def accuracy(logits, labels):
    """Top-1 accuracy, scalar"""
    # TODO
    pass

def cross_entropy_grad(logits, labels):
    """
    Gradient of mean cross-entropy loss w.r.t. logits.
    Returns: (N, C) — same shape as logits
    """
    # TODO: (softmax(logits) - one_hot(labels)) / N
    # You'll need to build the one-hot matrix
    pass

In [ ]:
# --- ASSERTS ---
probs = softmax(logits)
assert probs.shape == (N, C), 'softmax shape wrong'
assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-6), 'softmax rows dont sum to 1'
assert np.all(probs >= 0), 'negative probability'

# Stability check
big_logits = np.array([[1000., 1001., 1002.]])
stable_out = softmax(big_logits)
assert not np.any(np.isnan(stable_out)), 'stable softmax has NaN on large logits'
unstable_out = softmax_unstable(big_logits)
assert np.any(np.isnan(unstable_out)), 'unstable softmax should have NaN on [1000,1001,1002]'
print(f'  stable:   {stable_out}')
print(f'  unstable: {unstable_out}  ← expected NaN')

# Loss sanity: uniform logits → loss ≈ log(C)
uniform_logits = np.zeros((100, C))
uniform_labels = np.zeros(100, dtype=int)
loss_uniform = cross_entropy_loss(uniform_logits, uniform_labels)
assert np.isclose(loss_uniform, np.log(C), atol=1e-5), f'Uniform loss should be log({C})={np.log(C):.4f}, got {loss_uniform:.4f}'

# Accuracy: perfect prediction
perfect_logits = np.eye(C)[labels] * 100  # very high logit at correct class
assert np.isclose(accuracy(perfect_logits, labels), 1.0), 'Perfect logits should give 100% accuracy'

# Gradient shape
grad = cross_entropy_grad(logits, labels)
assert grad.shape == logits.shape, 'Gradient shape mismatch'

print('P6 PASSED ✓')
print(f'  Loss: {cross_entropy_loss(logits, labels):.4f}')
print(f'  Accuracy: {accuracy(logits, labels)*100:.1f}%')

---
## P7 — Loss Functions Zoo

Implement each loss + its gradient w.r.t. `y_pred`.

1. **MSE**: `mean((y_pred - y_true)²)` — gradient is `2*(y_pred - y_true)/N`
2. **Binary cross-entropy**: expects sigmoid'd predictions (so apply sigmoid to `logits_bin` first)  
   `mean(-[y*log(p + eps) + (1-y)*log(1-p + eps)])` — eps=1e-7 for stability
3. **Hinge loss**: for `y ∈ {-1, +1}` — `mean(max(0, 1 - y * ŷ))`  
   Convert `labels_bin` to {-1, +1} before using
4. **Huber loss** (delta=1.0):  
   - quadratic: `0.5*(error)²` when `|error| <= delta`
   - linear: `delta*(|error| - 0.5*delta)` otherwise

In [ ]:
def mse_loss(y_pred, y_true):
    # TODO
    pass

def mse_grad(y_pred, y_true):
    # TODO
    pass

def bce_loss(logits, labels, eps=1e-7):
    """
    logits: (N,) raw scores
    labels: (N,) in {0, 1}
    Returns: scalar
    """
    # TODO: apply sigmoid, then compute BCE
    pass

def bce_grad(logits, labels):
    """
    Gradient w.r.t. logits (not probs) — clean formula: (sigmoid(logits) - labels) / N
    """
    # TODO
    pass

def hinge_loss(scores, labels_01):
    """
    scores:     (N,) real-valued predictions
    labels_01:  (N,) in {0, 1} — will be converted to {-1, +1} inside
    Returns: scalar
    """
    # TODO: convert labels, compute max(0, 1 - y*ŷ)
    pass

def huber_loss(y_pred, y_true, delta=1.0):
    # TODO: use np.where to switch between quadratic and linear
    pass

def huber_grad(y_pred, y_true, delta=1.0):
    # TODO
    pass

In [ ]:
# --- ASSERTS ---

# MSE: perfect prediction -> 0 loss
assert np.isclose(mse_loss(y_true, y_true), 0.0), 'MSE of perfect pred should be 0'

# BCE: balanced random prediction on large batch → roughly log(2)
np.random.seed(1)
rand_logits = np.random.randn(10000)
rand_labels = np.random.randint(0, 2, 10000).astype(float)
bce_val = bce_loss(rand_logits, rand_labels)
assert 0.5 < bce_val < 1.0, f'BCE on random logits should be near log(2)≈0.693, got {bce_val:.4f}'

# Hinge: correct predictions with large margin → 0 loss
scores_correct = np.array([2.0, 3.0, 1.5])
labels_correct = np.array([1, 1, 1])
assert np.isclose(hinge_loss(scores_correct, labels_correct), 0.0), 'Large-margin correct preds → 0 hinge'

# Huber: small errors → MSE region
small_errors = np.array([0.1, -0.2, 0.3])
h = huber_loss(small_errors, np.zeros(3), delta=1.0)
m = mse_loss(small_errors, np.zeros(3))
assert np.isclose(h, 0.5 * m, atol=1e-6), 'Huber should equal 0.5*MSE for |error|<delta'

# Gradient shapes
assert mse_grad(y_pred, y_true).shape == y_pred.shape
assert bce_grad(logits_bin, labels_bin).shape == logits_bin.shape
assert huber_grad(y_pred, y_true).shape == y_pred.shape

print('P7 PASSED ✓')

In [ ]:
# Visualize loss behavior as a function of error
errors = np.linspace(-3, 3, 300)
zeros = np.zeros(300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(errors, 0.5 * errors**2, label='MSE (0.5*e²)', linewidth=2)
axes[0].plot(errors, [huber_loss(np.array([e]), zeros[:1]) for e in errors], label='Huber', linewidth=2, linestyle='--')
axes[0].set_title('MSE vs Huber')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(errors, np.maximum(0, 1 - errors), label='Hinge', linewidth=2, color='tomato')
axes[1].axvline(1, color='gray', linestyle=':', label='margin')
axes[1].set_title('Hinge Loss (y=+1)')
axes[1].legend()
axes[1].grid(alpha=0.3)

p_vals = np.linspace(1e-6, 1-1e-6, 300)
axes[2].plot(p_vals, -np.log(p_vals), label='-log(p), y=1', linewidth=2)
axes[2].plot(p_vals, -np.log(1 - p_vals), label='-log(1-p), y=0', linewidth=2, linestyle='--')
axes[2].set_title('Binary Cross-Entropy')
axes[2].legend()
axes[2].set_ylim(0, 5)
axes[2].grid(alpha=0.3)

plt.suptitle('Loss Functions')
plt.tight_layout()
plt.show()